# Build the equalization cache

Downloads Tiny-GenImage shard by shard, decodes each row **once**, and writes all four
equalization strategies from that single decode. Output lands on Drive as
`<cache>/<strategy>/<split>-<shard>.npy` plus one `.npz` of metadata per shard.

Run the cells top to bottom, including the pilot: a color-channel swap
or a transposed axis costs five minutes to catch there and 28 ruined runs to catch later.

Totals: validation 7,000 rows over 4 shards, train 28,000 over 14. Four strategies x 35,000
rows, plus the retained source parquets and the native validation blobs.

## Setup

In [ ]:
# Re-run this after any runtime restart.
#
# HF_XET_HIGH_PERFORMANCE has to be set before anything imports huggingface_hub, which
# latches it into its constants at import time. Every build cell below is a fresh
# `!python` subprocess and inherits this environment at launch, so setting it here is
# enough. (hf_transfer was the pre-Xet mechanism; it is deprecated and now ignored.)
!pip -q install hf_xet
import os
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

from google.colab import drive, userdata
drive.mount('/content/drive')

# Anonymous Hub requests are rate-limited first.
# Add a read token as a Colab secret named HF_TOKEN
# (key icon in the sidebar, then switch Notebook access on). Optional.
try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("HF token loaded")
except Exception as e:
    print("unauthenticated, continuing:", type(e).__name__)

DRIVE = "/content/drive/MyDrive/university/deep_learning"
CACHE = f"{DRIVE}/cache"

# Fail on a mistyped path now rather than three hours into the build.
assert os.path.isdir(DRIVE), f"not found: {DRIVE}"
print("cache ->", CACHE)

## Get the code

In [ ]:
# Idempotent: clones on the first run, pulls on every later one.
![ -d /content/deep-learning ] || git clone -q https://github.com/Ido-Josephsberg/deep-learning.git /content/deep-learning
%cd /content/deep-learning
!git pull --ff-only
!git log --oneline -1

## Pilot: two shards, then look at them

Everything downstream depends on `equalize` being right, and no assertion can see a BGR
swap. Build two shards, run the mechanical checks, then look at the contact sheet.

In [ ]:
!python -m src.data --split validation --max-shards 2 --keep-native --out-dir "{CACHE}"

In [ ]:
!python -m src.pilot --cache-dir "{CACHE}" --split validation --out-png "{DRIVE}/pilot_contact.png"
from IPython.display import Image; Image(f"{DRIVE}/pilot_contact.png")

## Full build

In [ ]:
# Validation first: every reported number comes from it, and it carries the metadata needed
# downstream. The .done markers make both commands resumable - they skip the two pilot
# shards, and a disconnect costs at most one shard, so just re-run the cell.
!python -m src.data --split validation --keep-native --out-dir "{CACHE}"
!python -m src.data --split train --out-dir "{CACHE}"

## Verify the finished cache


In [ ]:
import glob

# Exact expected bytes: rows * 128*128*3, plus the 128-byte .npy header. Anything that
# stopped mid-write lands on a size that is not in the expected set.
for strategy in ["center_crop", "random_crop", "rescale", "pad"]:
    for split, n_shards, per_shard in [("validation", 4, 1750), ("train", 14, 2000)]:
        files = sorted(glob.glob(f"{CACHE}/{strategy}/{split}-*.npy"))
        sizes = sorted({os.path.getsize(f) for f in files})
        ok = len(files) == n_shards and sizes == [per_shard * 128 * 128 * 3 + 128]
        print(f"{'OK ' if ok else 'BAD'} {strategy:<12}{split:<11} {len(files)}/{n_shards} files  {sizes}")

print("markers:", len(glob.glob(f"{CACHE}/markers/*.done")), "expect 18")

In [ ]:
!python -m src.pilot --cache-dir "{CACHE}" --split train --out-png "{DRIVE}/pilot_train.png"
!python -m src.pilot --cache-dir "{CACHE}" --split validation --out-png "{DRIVE}/pilot_val.png"